# Process GESTALT barcodes from serially sampled sperm data 
##### Authors: 
###### Step A: Connor Shrader and Andy Sposato
###### Step B: Connor Shrader and Andy Sposato
###### Step C: Connor Shrader and Andy Sposato
###### Step D: Connor Shrader and Andy Sposato
###### Step E: Connor Shrader, Andy Sposato & Kimberly Truong
######
### A) Throw out whole files that have low total reads and write passed-samples-info.csv.
##### Files that pass this step are given the extension: .passReadCounts. 
##### The passed-samples-info.csv is generated using only the .passReadCounts files. This csv will be important for step E when making the barcode matrix. 
### B) Filter barcodes by reads and proportion cutoffs. 
##### Barcodes that pass this step are added to files given the extension: .filteredReadCounts.  
### C) Examine questionable barcodes and identify contaminant barcodes. 
##### To examine barcodes for each fish, a dictionary of barcodes was created for each fish. 
##### Fish dictionaries are written to files and stored in the "fish_dictionaries" folder which is at the same level as SpermAnalysis_Muller1. 
##### The list of questionable and good barcodes were written to files in the "fish_dictionaries" folder. 
### D) Remove contaminant barcodes from their read files. 
##### Barcodes that pass this step are added to files given the extension: .filteredQualityReadCounts.
### E) Make barcode matrix. 
##### A composite barcode matrix file containing barcode information for all fish in the dataset is written to the experiments folder and used for generating Muller plots.
##### Barcode matrix files for each individual fish are also written to the experiments folder.
## Warnings: 
##### Make sure this python notebook file is stored at the same level as "SpermAnalysis_Muller1".
##### This pipeline will break if you change your naming system for sample replicates or mess with the data structure Kimberly established. See github.com/Gagnon-lab/Sperm-Analysis for Kimberly's pipeline.

### Settings

Step (A):
* `total_read_cutoff`: ignore any `.allReadCounts` files with total reads strictly less than this value.
* `start_age`: the age that fish start getting sampled. This is used to create entries in `passed_samples_info.csv`.

Step (B):
* `reads_cutoff`: remove a barcode from a sample if it has strictly less than this many reads.
* `prop_cutoff`: remove a barcode from a sample if it represents less than this proportion of the total reads.

Step (C):
* `run_complete_triple_rep_filter`: remove a barcode in all fish if it never appears in all 3 replicates for any choice of fish/sample.
* `run_fish_triple_rep_filter`: remove a barcode from a fish if it never appears in all 3 replicates in any sample (if set to `True`, then the value of `run_complete_triple_rep_filter` does not matter)
* `fish_one_sample_filter`: remove a barcode from a fish if it only appears in one time sample.
* `ratio_cutoff`: Any barcodes with a ratio score larger than this value are immediately assigned to the "winner" fish. Set this value to `False` to disable the ratio filter.
* `list_questionable_barcodes_before_classify`: Output a text file listing questionable barcodes before the classifying step.
* `list_questionable_barcodes_after_classify`: Output a text file listing barcodes that are *still* questionable after the classifying step.
* `image_questionable_barcodes_before_classify`: Plot a heatmap of all questionable barcodes before the classifying step.
* `image_questionable_barcodes_after_classify`: Plot a heatmap of all barcodes that are *still* questionable after the classifying step.
* `manual_classification`: a dictionary mapping barcodes to assigned outcomes. Ideally, this should be used only for debugging and testing. For example, setting `manual_classification = {'NONE_ABC_NONE': 'fish9'}` will assign fish9 as the "winner" fish. One can also set the value `'contaminant'` to remove the barcode.
* `delete_still_questionable_barcodes`: Whether to delete any barcodes that are still questionable after classification.

Step (E):
* `edited_fish_only`: If `False`, barcode matrices of control fish will be created, too.
* `fillna`: Whether to fill in empty values with 0's in the output barcode matrices.

In [1]:
# Step (A)
dataset = 'full-barcoded-fish'
total_read_cutoff = 1000
start_age = 4

# Step (B)
reads_cutoff = 10
prop_cutoff = 0.005

# Step (C)
run_complete_triple_rep_filter = True
run_fish_triple_rep_filter = True
run_fish_one_sample_filter = True
ratio_cutoff = 50

list_questionable_barcodes_before_classify = True
list_questionable_barcodes_after_classify = True
image_questionable_barcodes_before_classify = True
image_questionable_barcodes_after_classify = True

run_manual_classification = True
manual_classification = {}
# manual_classification = {
#     '3D+35_3D+64_NONE_81D+119_81D+119_81D+119_81D+119_NONE_1I+254+C_NONE': 'fish9'
# }

delete_still_questionable_barcodes = True

# Step (E)
edited_fish_only = False
fillna = False

### Import packages, set up directory, and define helper functions

In [2]:
import datetime
import os 
import re
import operator
import collections
import numpy as np
import pandas as pd
from natsort import natsorted 
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from textwrap import wrap

Make sure that we are in the current working directory (`process_sperm_barcodes/`)

In [3]:
cwd = os.getcwd()
if cwd.split('/')[-1] != 'sperm-data-analysis':
    raise Exception(f'Make sure you are running Python in the process_sperm_barcodes ' +
                    f'directory. You are currently running in {cwd}.')
else: print(f'We appear to be in the current working directory: {cwd}/.')

We appear to be in the current working directory: /Users/connor/Documents/github/sperm-data-analysis/.


Create a unique experiments subfolder to contain all the outputs from running the pipeline.

In [4]:
date = datetime.datetime.today().strftime('%m.%d.%Y') + "_" + \
       datetime.datetime.today().strftime('%I.%M_%p')

experiments_subfolder = "./experiments/" + date + "/"
                            
# if the experiments_subfolder doesn't already exist, create it
if not os.path.exists(experiments_subfolder): 
    os.mkdir(experiments_subfolder)

print(f'All outputs will be sent to {experiments_subfolder}.')

All outputs will be sent to ./experiments/07.02.2025_09.48_AM/.


Print the settings used to a text file titled `pipeline_settings.txt`.

In [5]:
summary = open(experiments_subfolder + "pipeline_settings.txt", "w")

def write_setting_line(file, var, name, buff):
    """
    Write a line of setting information to a file.

    'file': the file object we are writing to.
    'var':  the variable whose value we are storing.
    'name': the name of the variable to be written in the file.
    'buff': the buffer between the variable name and its value;
            used to evenly space information.
    
    """
    file.write(f"\t{(name + ' ').ljust(buff, '.')} {var}\n")
        
summary.write(f"Settings used during the pipeline run [{date}].\n\n")

summary.write("Step (A): throw out files with low read counts.\n")

buff = 21
write_setting_line(summary, total_read_cutoff, 'total_read_cutoff', buff)
write_setting_line(summary, start_age, 'start_age', buff)

summary.write("\nStep (B): filter barcodes.\n")
write_setting_line(summary, reads_cutoff, 'reads_cutoff', buff)
write_setting_line(summary, prop_cutoff, 'prop_cutoff', buff)

buff = 34
summary.write("\nStep (C): identify contaminant barcodes.\n")
write_setting_line(summary, run_complete_triple_rep_filter, 'run_complete_triple_rep_filter', buff)
write_setting_line(summary, run_fish_triple_rep_filter, 'run_fish_triple_rep_filter', buff)
write_setting_line(summary, run_fish_one_sample_filter, 'run_fish_one_sample_filter', buff)
write_setting_line(summary, ratio_cutoff, 'ratio_cutoff', buff)
summary.write('\n')

buff = 50
write_setting_line(summary, list_questionable_barcodes_before_classify, 'list_questionable_barcodes_before_classify', buff)
write_setting_line(summary, list_questionable_barcodes_after_classify, 'list_questionable_barcodes_after_classify', buff)
write_setting_line(summary, image_questionable_barcodes_before_classify, 'image_questionable_barcodes_before_classify', buff)
write_setting_line(summary, image_questionable_barcodes_after_classify, 'image_questionable_barcodes_after_classify', buff)

buff = 34
summary.write('\n')
write_setting_line(summary, run_manual_classification, 'run_manual_classification', buff)
summary.write('\n')
if run_manual_classification:
    summary.write(f"\tManual classifications were as follows:\n")
    for key, value in manual_classification.items():
        buff = 4 + max([len(key) for key in manual_classification.keys()])
        # summary.write(f"\t{key}: {value}\n")
        write_setting_line(summary, value, key, buff)
    summary.write("\n")

buff = 38
write_setting_line(summary, delete_still_questionable_barcodes, 'delete_still_questionable_barcodes', buff)

buff = 20
summary.write("\nStep (E): create barcode matrices.\n")
write_setting_line(summary, edited_fish_only, 'edited_fish_only', buff)
write_setting_line(summary, fillna, 'fillna', buff)
summary.close() 

In [6]:
def directory(fish, sample, replicate):
    """
    This function takes in an identifier for the fish, sample, and replicate,
    and returns a string representing the location of files for that sample.

    For example, calling directory('fish1', '2', '3') will return
    './SpermAnalysis_Muller1/data/viz-Output/fish1_s2_rep3/fish1_s2_rep3'.
    """
    filename = f"{fish}_s{sample}_rep{replicate}"
    return f"./data/{dataset}/{filename}/{filename}"

def tuple_from_file(filepath):
    """
    This function takes in a filepath and returns the fish, sample, and replicate
    that correspond to it. It is, in a sense, the inverse function of directory.

    For example, calling
    directory('./SpermAnalysis_Muller1/data/viz-Output/fish1_s2_rep3/fish1_s2_rep3.allReadCounts')
    will return ('fish1', '2', '3').
    """
    fish, sample, replicate = (filepath.split("/")[-1]).split('.')[0].split('_')
    return fish, sample[1:], replicate[3:]

In [7]:
def summarize_barcode(data, barcode):
    """
    This function inputs a DataFrame ('data') of reads, and a barcode. It returns
    a DataFrame ('group_fish') that contains the total number of reads for the
    given barcode in each fish. In addition, this function returns the 'ratio'
    used to identify winner fish.
    """
    single_barcode = data[data.barcode == barcode].drop("barcode", axis = 1)
    group_fish = single_barcode.groupby(['fish'], observed = True, as_index = False).aggregate({'reads': 'sum'})
    group_fish.sort_values('reads', ascending = False, inplace = True)

    winner_reads = group_fish.reads.iloc[0]
    total_reads = group_fish.reads.sum()

    ratio = winner_reads / (total_reads - winner_reads)

    return group_fish, ratio

def heatmap(data, barcode):
    """
    This function inputs a DataFrame ('data') of reads and a barcode. It plots a heatmap showing
    the reads of that barcode across all fish, samples, and replicates.
    """
    data['sample'] = pd.to_numeric(data['sample'])
    group_fish, ratio = summarize_barcode(data, barcode)

    # sort fish in reverse order so that fish with large reads are on top.
    idx = group_fish.index#[::-1]
    
    single_barcode = data[data.barcode == barcode].drop("barcode", axis = 1)
    single_barcode["rep_count"] = 1
    group_samples = single_barcode.groupby(['fish', 'sample'], as_index = False, observed = True) \
                             .aggregate({'rep_count': 'sum', 'reads': 'sum'})
    
    to_plot = group_samples.pivot(index = 'fish', columns = 'sample', values = 'rep_count') \
                           .fillna(0).iloc[idx]
    
    read_counts = group_samples.pivot(index = "fish", columns = "sample", values = "reads") \
                               .fillna(0).astype(int).iloc[idx]

    # the maximum number of replicates that appear in any fish/sample for this barcode
    max_rep = int(to_plot.values.max())

    # [black, red, green, blue]
    myColors = ((0.0, 0.0, 0.0, 1.0), (0.8, 0.0, 0.0, 1.0), (0.0, 0.8, 0.0, 1.0), (0.0, 0.0, 0.8, 1.0))
    cmap = LinearSegmentedColormap.from_list('Custom', myColors[:max_rep+1], max_rep+1)
    
    fig, ax = plt.subplots()
    ax = sns.heatmap(to_plot, annot = read_counts, annot_kws = {"size": 6}, fmt = 'd', cmap = cmap, vmin = 0, vmax = max_rep)
    # ax.invert_yaxis()
    ax.set_title(f"{barcode}, r={ratio:.2f}", fontsize = 8)
    
    colorbar = ax.collections[0].colorbar
    colorbar.set_ticks([max_rep/(max_rep+1)*i+max_rep/(2*(max_rep+1)) for i in range(max_rep + 1)])
    colorbar.set_ticklabels([f'{i}' for i in range(max_rep+1)])
    colorbar.set_label("Number of replicates the barcode appeared in")

    return fig, ax

In [8]:
def identify_questionable(data):
    """
    This function takes in a DataFrame of reads and identifies which barcodes are questionable.
    It returns a list, called questionable_barcodes, containing the names of barcodes that appear
    in multiple fish.
    """
    barcodes = data[data["barcode"] != unedited].barcode.unique()
    questionable_barcodes = []

    for barcode in barcodes:
        if len(data[data.barcode == barcode].fish.unique()) > 1:
            questionable_barcodes.append(barcode)

    return questionable_barcodes

## A) Throw out whole files that have low total reads and write passed-samples-info.csv.

Currently, Kimberly's code relies on the user writing a samples-info.csv from scratch. We want to filter samples that are totally garbage (all barcodes have small numbers of reads). Then write a samples-info.csv based on the .allReadCounts files that remain after that step. We'll have to rewrite .allReadCounts we want to keep with a "pass" key in the name so that we can direct next steps to read just those files. Next steps include removing barcodes with low read numbers and proportions and removing contaminant barcodes from their samples. 

#### Load all the .allReadCounts files in the data directory to `all_data`.

In [9]:
# Get list of file paths leading to .allReadCounts files
# At the same time, save the corresponding path "tuples" to a list.
# This will be used later to detect "empty" .allReadCounts files
# and output the correct values to the step A summary file.

filepaths = []
for dirpath, dirnames, filenames in os.walk(f"data/{dataset}/"):
   for filename in filenames:
      if filename.endswith(".allReadCounts"):
          filepath = os.path.join(dirpath, filename)
          filepaths.append(filepath)

filepaths = natsorted(filepaths)
all_file_tuples = [tuple_from_file(fp) for fp in filepaths]
print(f"An example .allReadCounts filepath in your directory:\n{filepaths[0]}.")
print(f"An example file tuple: {all_file_tuples[0]}.")
print(f"\nIn total, {len(filepaths)} .allReadCounts files were found.")

An example .allReadCounts filepath in your directory:
data/full-barcoded-fish/fish1_s1_rep1/fish1_s1_rep1.allReadCounts.
An example file tuple: ('fish1', '1', '1').

In total, 1272 .allReadCounts files were found.


In [10]:
# Load all of the reads from .allReadCounts files to a pandas DataFrame.

# data is a list that will contain one data frame per file.
df_list = []

# Each file tuple (such as ('fish1', '1', '1')) will be assigned to either kept_groups
# (if the read count is sufficiently large), removed_groups (if it is not sufficiently large),
# or empty_groups (if the file is empty). We also track the number of reads per group, which
# is output in the Step A summary report.
kept_groups = []
removed_groups = []
empty_groups = []
reads_per_group = {}

for fish, sample, replicate in all_file_tuples:
    # Read the .allReadCounts file as if it's a TSV file.
    read_file = directory(fish, sample, replicate) + '.allReadCounts'
    df = pd.read_csv(read_file, sep = '\t', index_col = 'array')

    # Edit the data frame and append it to our list of data frames.
    df.rename(columns = {'event': 'barcode', 'count': 'reads'}, inplace = True)
    df.index.name = None
    df.insert(loc = 1, column = 'fish', value = fish)
    df.insert(loc = 2, column = 'sample', value = sample)
    df.insert(loc = 3, column = 'replicate', value = replicate)
    df.insert(loc = 4, column = 'file', value = f"{fish}_s{sample}_rep{replicate}")

    # Determine whether the file passes our initial filter.
    reads = df.reads.sum()
    reads_per_group[(fish, sample, replicate)] = reads
    if reads >= total_read_cutoff:
        df.insert(loc = 7, column = 'pass', value = True)
        kept_groups.append((fish, sample, replicate))
    else:
        df.insert(loc = 7, column = 'pass', value = False)
        removed_groups.append((fish, sample, replicate))

    # If the file is non-empty, append it to our data list.
    if len(df) > 0:
        df_list.append(df)
    else: empty_groups.append((fish, sample, replicate))

In [11]:
print(f'Of the {len(filepaths)} total files, {len(kept_groups)} passed ' + 
      f'the initial filter (they had over {total_read_cutoff} reads).')
      
print(f'Of the {len(removed_groups)} files that did not pass, {len(empty_groups)} was completely empty.')

Of the 1272 total files, 1158 passed the initial filter (they had over 1000 reads).
Of the 114 files that did not pass, 0 was completely empty.


In [12]:
# Merge the list of data frames into a single data frame.
all_data = pd.concat(df_list, ignore_index = True)
print(f'The number of entries in all_data is {len(all_data)}. The first five rows are below:')
all_data.head()

The number of entries in all_data is 1023042. The first five rows are below:


,barcode,fish,sample,replicate,file,reads,proportion,pass
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True


In [13]:
# Save all_data to a file.
all_data.to_csv(experiments_subfolder + 'all_data.csv', index = False)

In [14]:
all_data.barcode[0]

'12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169_84D+169_84D+169_NONE'

#### Write `passed-samples-info` file.

In [15]:
sample_info = []
for fish, sample, replicate in kept_groups:
    sample_id = f'{fish}_s{sample}_rep{replicate}'
    sample_info.append({'sample_id': sample_id, 'fish': fish[4:], 
                        'age': int(sample) + start_age, 'replicate': replicate, 
                        'condition': 'control' if 'ctrl' in fish else 'edited'})
    
passed_samples_info = pd.DataFrame(sample_info)
passed_samples_info.to_csv(experiments_subfolder + 'passed_samples_info.csv', index = False)
passed_samples_info.head()

,sample_id,fish,age,replicate,condition
0,fish1_s1_rep1,1,5,1,edited
1,fish1_s1_rep2,1,5,2,edited
2,fish1_s1_rep3,1,5,3,edited
3,fish1_s2_rep1,1,6,1,edited
4,fish1_s2_rep2,1,6,2,edited


#### Step A summary report.

In [16]:
all_groups = kept_groups + removed_groups

summary = open(experiments_subfolder + "stepA_summary.txt", "w")
summary.write("Summary of step A: Throw out whole files that have low total reads and write passed-samples-info.csv.\n\n")
summary.write("Your data contains " + str(len(all_groups)) + ' .allReadCounts files, or ' + str(len(all_groups)) + ' samples including replicates.\n\n')
summary.write("If a file contained at least " + str(total_read_cutoff) + " total reads across all barcodes, then it has passed this filter step.\n") 
summary.write("\n")
if len(removed_groups) == 0: 
    summary.write("All of your files passed this step.\n")
elif len(removed_groups) > 0:
    summary.write(f'{len(removed_groups)} files failed this step:\n')

    for group in removed_groups: 
        fish, sample, replicate = group
        sample_id = f'{fish}_s{sample}_rep{replicate}'
        summary.write(f'{sample_id} failed with only {reads_per_group[group]} total reads. \n')
            
summary.write("\n")
summary.write(f'{len(kept_groups)} files passed the filter.\n\n')

summary.write("The contents of read counts from all files was written in a new file titled 'all_data.csv.\n")
summary.write("The contents of read counts that passed were copied to a new file titled 'pass_data.csv' \nin the same subfolder.\n\n")
summary.write("The passed-samples-info.csv was also written and saved to the folder called data.\nIt contains only samples that pass this step.\n")
summary.close() 

## B) Filter barcodes by reads and proportion cutoffs.

#### Apply the filters and summarize results

In [17]:
pass_data = all_data[all_data['pass']].copy()
pass_data.drop('pass', axis = 1, inplace = True)

# pass_data['reads_filter'] = pd.NA
# pass_data.loc[pass_data['pass'], 'reads_filter'] = pass_data['reads'] >= reads_cutoff
pass_data['reads_filter'] = pass_data['reads'] >= reads_cutoff

# pass_data['prop_filter'] = pd.NA
# pass_data.loc[pass_data['pass'], 'prop_filter'] = pass_data['proportion'] >= prop_cutoff
pass_data['prop_filter'] = pass_data['proportion'] >= prop_cutoff

filtered_data = pass_data[pass_data['reads_filter'] & pass_data['prop_filter']].copy()
filtered_data.drop(['reads_filter', 'prop_filter'], axis = 1, inplace = True)

In [18]:
# Track how many reads/barcodes were removed for each sample
pass_aggregate = pass_data.groupby(by = ['fish', 'sample', 'replicate'])['reads'].agg(['size', 'sum'])
pass_aggregate.rename(columns = {'size': 'pass_size', 'sum': 'pass_sum'}, inplace = True)

filtered_aggregate = filtered_data.groupby(by = ['fish', 'sample', 'replicate'])['reads'].agg(['size', 'sum'])
filtered_aggregate.rename(columns = {'size': 'filtered_size', 'sum': 'filtered_sum'}, inplace = True)

step_b_summary = pd.concat([pass_aggregate, filtered_aggregate], axis = 1)
step_b_summary.head()

pass_size  pass_sum  filtered_size  filtered_sum
fish  sample replicate                                                  
fish1 1      1                710     76693              6         72943
             2                451     40635              8         38979
             3                630     52249              6         49230
      10     1                448     40540             12         37813
             2                464     41842             11         38952

#### Write `pass_data.csv`.

In [19]:
pass_data.to_csv(experiments_subfolder + 'pass_data.csv', index = False)

#### Step B summary report

In [20]:
summary = open(experiments_subfolder + "stepB_summary.txt", 'w')
summary.write("Summary of step B: Filter barcodes by reads and proportion cutoffs.\n\n")
summary.write("For each .passReadCounts file, if a barcode had fewer than " + str(reads_cutoff) + " reads or a proportion below " + str(prop_cutoff) + ",\nit was removed from the file.\n\n")
summary.write("A list of barcodes that were removed was written to a sample file that ends with 'stepB_removed_barcodes.txt'\nThis may be useful to verify that the changes from .passReadCounts to .filteredReadCounts were accurate.\n\n")
summary.write("Here is a recap of the filtering for each sample: \n\n") 
summary.write("**************************************************************************************** \n\n")

filepath_string = './SpermAnalysis_Muller1/data/viz-Output/' 

for group in step_b_summary.index: 
    fish, sample, replicate = group
    sample_id = f'{fish}_s{sample}_rep{replicate}'

    pass_read_counts = step_b_summary.loc[group, "pass_sum"]
    pass_count = step_b_summary.loc[group, "pass_size"]
    filtered_read_counts = step_b_summary.loc[group, "filtered_sum"]
    filtered_count = step_b_summary.loc[group, "filtered_size"]
    
    reads_removed = pass_read_counts - filtered_read_counts
    summary.write(sample_id + " had " + str(pass_read_counts) + " reads in the .passReadCounts file. Filtering removed " + str(reads_removed) + ' reads.\n')
    summary.write("There are now " + str(filtered_read_counts) + " reads in the " + sample_id + ".filteredReadCounts file.\n\n")
    
    barcodes_removed = pass_count-filtered_count
    summary.write(sample_id + " had " + str(pass_count) + " barcodes in the .passReadCounts file. Filtering removed " + str(barcodes_removed) + ' barcodes.\n')
    if filtered_count == 1: 
        summary.write("There is now " + str(filtered_count) + " barcode in the " + sample_id + ".filteredReadCounts file.\n\n")
    else:     
        summary.write("There are now " + str(filtered_count) + " barcodes in the " + sample_id + ".filteredReadCounts file.\n\n")
    summary.write("**************************************************************************************** \n\n")
summary.close()

## C) More filters

In [21]:
new_columns = ['complete_triple_rep_classification', 'fish_triple_rep_classification',
               'fish_one_sample_classification']

for new_column in new_columns:
    filtered_data.loc[:, new_column] = True
    filtered_data[new_column] = filtered_data[new_column].astype('bool')

Determine values of `complete_triple_rep_classification`, `fish_triple_rep_classification`, and `fish_one_sample_classification`.

In [22]:
fd = filtered_data
for barcode in fd['barcode'].unique():
    single_barcode = filtered_data[filtered_data.barcode == barcode]
    grouped = single_barcode.groupby(['fish', 'sample'], observed = True, as_index = False)['replicate'].count()
    
    # grouped is a DataFrame with columns (fish, sample, replicate). The value of 'replicate' is
    # the count of replicates in which there is at least one read for that fish and sample.
    if grouped.replicate.max() < 3 and run_complete_triple_rep_filter:
        fd.loc[fd.barcode == barcode, "complete_triple_rep_classification"] = False
        # filtered_data.loc[filtered_data[filtered_data.barcode == barcode].index, "complete_triple_rep_classification"] = False
        
    for fish in single_barcode.fish.unique():
        # grouped is a DataFrame with columns (sample, replicate). The value of 'replicate' is the count
        # of replicates in which there is at least one read for that sample.
        grouped = single_barcode[single_barcode.fish == fish].groupby(['sample'], as_index = False)['replicate'].count()
        if grouped.replicate.max() < 3 and run_fish_triple_rep_filter:
            # never appears in 3 reps, throw out
            fd.loc[(fd.barcode == barcode) & (fd.fish == fish), "fish_triple_rep_classification"] = False
            # fd.loc[fd[(filtered_data.barcode == barcode) & (filtered_data.fish == fish)].index, "fish_triple_rep_classification"] = False
            
        if len(grouped) == 1 and run_fish_one_sample_filter:
            # only appears in one sample, throw out
            fd.loc[(fd.barcode == barcode) & (fd.fish == fish), "fish_one_sample_classification"] = False
            # filtered_data.loc[filtered_data[(filtered_data.barcode == barcode) & (filtered_data.fish == fish)].index, "fish_one_sample_classification"] = False

In [23]:
# fd_pc is the data that passes the initial classification steps
fd2 = fd[fd[['complete_triple_rep_classification', 'fish_triple_rep_classification',
               'fish_one_sample_classification']].all(axis=1)]

# Make sure that we just sampled the correct columns
print('We will now ignore any rows that contain False in any of the columns: ' + ', '.join(fd.columns[7:10]) + '.\n')

print(f'Before initial classification, there were {len(fd)} rows. After initial classification, there are now {len(fd2)}.')

fd2.head()

We will now ignore any rows that contain False in any of the columns: complete_triple_rep_classification, fish_triple_rep_classification, fish_one_sample_classification.

Before initial classification, there were 12301 rows. After initial classification, there are now 9560.


,barcode,fish,sample,replicate,file,reads,proportion,complete_triple_rep_classification,fish_triple_rep_classification,fish_one_sample_classification
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True,True,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True,True,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True,True,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True,True,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True,True,True


## C) Examine questionable barcodes and identify contaminant barcodes. 

Questionable barcodes are barcodes that show up in multiple animals. They are likely contamination. We want to confidently remove them from samples where they do not belong. Before we can remove them from the appropriate files, we need to examine each according to the number of reads that support them in every fish. 

### Find questionable barcodes
###### These are barcodes that show up in multiple fish. 
###### A barcode will not be tagged questionable if it shows up in multiple samples or replicates of a single fish. 

In [24]:
# save the unedited barcode string as a variable
unedited = "NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE"

In [25]:
questionable_barcodes = identify_questionable(fd2)
print(f'There are {len(questionable_barcodes)} questionable barcodes.')
# filtered_data.head()

There are 28 questionable barcodes.


In [26]:
new_columns = ['manual_classification', 'ratio_classification']

for new_column in new_columns:
    fd2.loc[:, new_column] = True
    # fd2.loc[:, new_column] = fd2[new_column].astype('bool')

/var/folders/59/35nb2tq50vz3vpwjkwkzppvh0000gn/T/ipykernel_29237/2398964910.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fd2.loc[:, new_column] = True
/var/folders/59/35nb2tq50vz3vpwjkwkzppvh0000gn/T/ipykernel_29237/2398964910.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fd2.loc[:, new_column] = True


In [27]:
# Process any barcodes that have manual assignments set by the user.
for key, value in manual_classification.items():
    if value == 'contaminant':
        fd2.loc[fd2[fd2.barcode == key].index, "manual_classification"] = False
    else:
        fd2.loc[fd2[(fd2.barcode == key) & \
            (fd2.fish != value)].index, "manual_classification"] = False

In [28]:
# fd_pc is the data that passes the initial classification steps
fd_pc = fd2[fd2['manual_classification']]

print(f'Before initial classification, there were {len(fd2)} rows. After initial classification, there are now {len(fd_pc)}.')

fd_pc.head()

Before initial classification, there were 9560 rows. After initial classification, there are now 9560.


,barcode,fish,sample,replicate,file,reads,proportion,complete_triple_rep_classification,fish_triple_rep_classification,fish_one_sample_classification,manual_classification,ratio_classification
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True,True,True,True,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True,True,True,True,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True,True,True,True,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True,True,True,True,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True,True,True,True,True


In [29]:
still_questionable_barcodes_before_ratio = identify_questionable(fd_pc)
print(f'Before initial classification, {len(questionable_barcodes)} barcodes were questionable. ' +
      f'{len(manual_classification)} had manual assignment. {len(still_questionable_barcodes_before_ratio)} ' +
      f'are still questionable after the initial classification steps.')

Before initial classification, 28 barcodes were questionable. 0 had manual assignment. 28 are still questionable after the initial classification steps.


In [30]:
if ratio_cutoff is not False:
    for barcode in still_questionable_barcodes_before_ratio:
        one_barcode = fd_pc[fd_pc.barcode == barcode]
        
        aggregate = one_barcode.groupby(by = 'fish', as_index = False, observed = True) \
                                      .aggregate({'reads': 'sum'})
        
        aggregate.sort_values('reads', ascending = False, inplace = True, ignore_index = True)
        
        winner_reads = aggregate.reads.iloc[0]
        winner_fish = aggregate.fish.iloc[0]
        
        total_reads = aggregate.reads.sum()
        
        ratio = winner_reads / (total_reads - winner_reads)

        if ratio > ratio_cutoff:
            fd2.loc[fd2[(fd2.barcode == barcode) & \
                (fd2.fish != winner_fish)].index, "ratio_classification"] = False

In [31]:
fd2.to_csv(experiments_subfolder + 'filtered_data.csv', index = False)

In [32]:
classified_data = fd2[fd2[['manual_classification', 'ratio_classification']].all(axis=1)].copy()

In [33]:
still_questionable_barcodes = identify_questionable(classified_data)
print(f'There are {len(still_questionable_barcodes)} barcodes that are still questionable.')

There are 21 barcodes that are still questionable.


In [34]:
classified_data['no_longer_questionable'] = ~classified_data['barcode'].isin(still_questionable_barcodes)
classified_data.drop(columns=['complete_triple_rep_classification', 'fish_triple_rep_classification',
                              'fish_one_sample_classification', 'manual_classification', 'ratio_classification'], inplace = True)
classified_data.head()

,barcode,fish,sample,replicate,file,reads,proportion,no_longer_questionable
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True


In [35]:
classified_data.to_csv(experiments_subfolder + 'classified_data.csv', index = False)

Remove any still questionable barcodes if `delete_still_questionable_barcodes` is True.

In [36]:
final_data = None
if delete_still_questionable_barcodes:
    final_data = classified_data[classified_data['no_longer_questionable']]
else:
    final_data = classified_data

final_data.to_csv(experiments_subfolder + 'final_data.csv', index = False)

### Associate each barcode with a contamination status

In [37]:
questionable_barcodes_summary = dict()
for bc in questionable_barcodes:
    query = classified_data[classified_data.barcode == bc].fish.unique()
    
    if bc in manual_classification:
        questionable_barcodes_summary[bc] = f'{manual_classification[bc]} (manual classification)'
    elif bc in still_questionable_barcodes:
        questionable_barcodes_summary[bc] = "still questionable"
    elif len(query) == 0:
        questionable_barcodes_summary[bc] = "contaminant"
    elif len(query) == 1:
        fish = query[0]
        if bc in still_questionable_barcodes_before_ratio:
            questionable_barcodes_summary[bc] = f'{fish} (found by ratio)'
        else:
            questionable_barcodes_summary[bc] = f'{fish} (found before ratio)'
    else:
        raise Exception(f'barcode {bc} is weird.')

### Output findings

In [38]:
# write questionable barcodes to a file
if list_questionable_barcodes_before_classify:
    output = open(f'{experiments_subfolder}questionable_barcodes_list.txt', 'w')
    for barcode in questionable_barcodes: 
        output.write(f'{barcode}: {questionable_barcodes_summary[barcode]}')
        output.write("\n")
    output.close()

if list_questionable_barcodes_after_classify and len(still_questionable_barcodes) > 0:
    output = open(f'{experiments_subfolder}still_questionable_barcodes_list.txt', 'w')
    for barcode in still_questionable_barcodes: 
        output.write(barcode)
        output.write("\n")
    output.close()

In [39]:
# produce heatmaps
if image_questionable_barcodes_before_classify:
    directory = f'{experiments_subfolder}heatmaps_before_classification'
    if not os.path.exists(directory): 
        os.mkdir(directory)
    data = fd2.copy()
    for i, bc in enumerate(questionable_barcodes):
        _, ratio = summarize_barcode(data, bc)
        fig, ax = heatmap(data, bc)
        ax.set_title("\n".join(wrap(f"{bc}", 50) + [f"\nratio: {ratio:.2f}, outcome: {questionable_barcodes_summary[bc]}"]))
        fig.tight_layout()
        plt.savefig(f"{directory}/barcode[{i}].png")
        plt.close()

if image_questionable_barcodes_after_classify:
    directory = f'{experiments_subfolder}heatmaps_after_classification'
    if not os.path.exists(directory): 
        os.mkdir(directory)
    data = classified_data.copy()
    for i, bc in enumerate(questionable_barcodes):
        if bc not in still_questionable_barcodes: continue
        _, ratio = summarize_barcode(data, bc)
        fig, ax = heatmap(data, bc)
        ax.set_title("\n".join(wrap(f"{bc}", 50) + [f"\nratio: {ratio:.2f}, outcome: {questionable_barcodes_summary[bc]}"]))
        fig.tight_layout()
        plt.savefig(f"{directory}/barcode[{i}].png")
        plt.close()

### Step C summary report

In [40]:
# summary = open(experiments_subfolder+"stepC_summary.txt", 'w')
# summary.write("Summary of step C: Examine questionable barcodes and identify contaminant barcodes.\n\n")
# summary.write("A fish dictionary was created for each fish in your data and written to a file like fish1_dictionary.tsv \n\n")
# summary.write("A dictionary holds every barcode found in any sample or replicate belonging to that animal.\n")
# summary.write("For each barcode in that dictionary, information about:\n\t1) the sample replicate it showed up in,\n\t2) the proportion of that sample replicate's barcode population\n\t3) the number of reads associated with that barcode within the sample replicate\n")
# summary.write("All dictionaries were also compiled into one file called fishes_dictionaries.tsv.\n\n")
# summary.write("The fish dictionaries were used to compare barcode occurrence across fish.\nQuestionable barcodes are barcodes that show up in multiple fish.\nThis list was written to questionable_barcodes_list.txt.\n")
# summary.write("\nThe rest of this step is exploring those questionable barcodes. The next set of filtered files\nare generated in Step D.\n")
# summary.write("\nHere is an example fish dictionary: \n\n") 

# dict_file = open("fish_dictionaries/fishctrl3_dictionary.tsv", "r")
# for line in dict_file.readlines(): 
#     summary.write(line)
# dict_file.close()

# summary.write("\n")
# summary.close()

## D) Remove contaminant barcodes from read files.

The intent here is to remove barcodes that are contamination in just the fish where they don't belong. So for every questionable barcode, we want to categorize if it's a contaminant everywhere or has a source. 

### Remove contaminant barcodes from .filteredReadCount files
##### This will rewrite all of our .filteredReadCounts files as .filteredQualityReadCounts
###### Contaminant barcodes without a clear source are removed everywhere.
###### Contaminant barcodes with a clear source are removed everywhere except files belonging to the source fish.

In [41]:
classified_data.to_csv(experiments_subfolder + 'classified_data.csv', index = False)

The .filteredQualityReadCounts is written to the same subfolder where the original .allReadCounts file, .passReadCounts file and .filteredReadCounts file are. The .filteredQualityReadCounts should be used as the input in the make barcode matrix code. 

### Step D summary report

In [42]:
# filtered_quality_samples = []
# for dirpath, dirnames, filenames in os.walk("."):
#    for filename in filenames:
#       if filename.endswith(".filteredQualityReadCounts"):
#           sample_id = filename.split(".") # split file path by '.'
#           sample_id = str(sample_id[0]) # sample_id = full name of the sample (i.e. fishctrl3_s5_rep3)
#           filtered_quality_samples.append(sample_id)
# filtered_quality_samples_sorted = natsorted(filtered_quality_samples)

In [43]:
summary = open(experiments_subfolder+"stepD_summary.txt", 'w')
summary.write("Summary of step D: Remove contaminant barcodes from their read files.\n\n")
summary.write("The barcodes in questionable barcodes list were examined carefully for their presence across animals.\n")
summary.write("If a clear source of that barcode could be found, the questionable barcode was assigned a source fish.\n")
summary.write("If there was not a clear winner, the questionable barcode was considered contaminant everywhere.\n\n")
summary.write("For each questionable barcode, it was removed from the .filteredReadCounts files where it is considered contamination.\n")
summary.write("The new read files are written with the extension: .filteredQualityReadCounts\n\n")
summary.write("\nHere is a recap of the filtering for each sample: \n\n")
summary.write("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ \n\n")

# filepath_string = './SpermAnalysis_Muller1/data/viz-Output/'

def make_plural(array):
    return 's' if len(array) != 1 else ''

for fish, sample, replicate in kept_groups:
    filtered_slice = filtered_data.query(f"fish == '{fish}' & sample == '{sample}' & replicate == '{replicate}'")
    filtered_barcodes = filtered_slice.barcode.unique()
    
    classified_slice = classified_data.query(f"fish == '{fish}' & sample == '{sample}' & replicate == '{replicate}'")
    classified_barcodes = classified_slice.barcode.unique()

    barcodes_removed = set(filtered_barcodes) - set(classified_barcodes)
    reads_removed = filtered_slice.reads.sum() - classified_slice.reads.sum()

    sample_id = f'{fish}_s{sample}_rep{replicate}'

    summary.write(sample_id + " had " + str(filtered_slice.reads.sum()) + " reads in the .filteredReadCounts file. Filtering removed " + str(reads_removed) + ' reads.\n')
    summary.write("There are now " + str(classified_slice.reads.sum()) + " reads in the " + sample_id + ".filteredQualityReadCounts file.\n\n")

    summary.write(f'{sample_id} had {len(filtered_barcodes)} barcode{make_plural(filtered_barcodes)} ' +
                  f'in the .filteredReadCounts file. Filtering removed {len(barcodes_removed)} barcode{make_plural(filtered_barcodes)}.\n')

    summary.write(f'There {"is" if len(classified_barcodes) == 1 else "are"} {len(classified_barcodes)} barcode' +
                  f'{make_plural(classified_barcodes)} in the {sample_id}.filteredQualityReadCounts file.\n\n')
        
    if len(barcodes_removed) == 0: 
        summary.write("No barcodes were removed from " + sample_id + '.filteredReadCounts. \n')
    else:
        if len(barcodes_removed) == 1:
            summary.write("This barcode was removed from " + sample_id + '.filteredReadCounts: \n')
        else:
            summary.write("These barcodes were removed from " + sample_id + '.filteredReadCounts: \n')
        for barcode in barcodes_removed:
            if barcode in questionable_barcodes_summary: 
                if questionable_barcodes_summary[barcode] == 'contaminant': 
                    summary.write('\t' + barcode + ": promiscuous barcode\n" )
                else: 
                    summary.write('\t' + barcode + ": contamination from " + questionable_barcodes_summary[barcode] + "\n")    
    summary.write("\n")
    summary.write("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ \n\n")

summary.write("\n")
summary.close()

## E) Make barcode matrix.

The barcode matrix will be used as input when generating Muller plots in R. 

In [44]:
print(os.getcwd())

/Users/connor/Documents/github/sperm-data-analysis


### Assemble the matrix

In [45]:
# # load passed-sample-info.csv
# samples_df = pd.read_csv("./SpermAnalysis_Muller1/data/passed-samples-info.csv", sep=",", index_col=0, dtype={'replicate':'category'})
# # Andy commented out the next line so this code will read your control samples. To look at edited fish only, uncomment the next line:
# if edited_fish_only:
#     samples_df = samples_df[samples_df['condition']=='edited']
# print(samples_df)

In [46]:
d = final_data.copy()
d['sample'] = pd.to_numeric(d['sample'])
d['sample_id'] = d['fish'] + '_s' + d['sample'].apply(str) + '_rep' + d['replicate']
# classified_data.sort_values('sample', inplace = True)

In [47]:
directory = experiments_subfolder + 'matrix'
if not os.path.exists(directory): 
    os.mkdir(directory)

fish_list = d.fish.unique()
for fish in fish_list:
    fish_df = d[d['fish'] == fish].copy()
    pt = fish_df.pivot_table(columns = ['sample_id'], index = 'barcode', values = 'reads', sort = False)
    if fillna:
        pt.fillna(0, inplace = True)
    pt.to_csv(f'{directory}/{fish}_barcodeMatrix_3filters.tsv', sep = '\t')

pt = d.pivot_table(columns = ['sample_id'], index = 'barcode', values = 'reads', sort = False)
if fillna:
    pt.fillna(0, inplace = True)
pt.to_csv(f'{directory}/barcodeMatrix_3filters.tsv', sep = '\t')

In [48]:
# # the dataframe should already be ordered, but go ahead and run this
# # reorder dataframe rows by fish then date 
# samples_df.sort_values(by=['fish','age'], inplace=True)
# ind_sorted = samples_df.index
# # you can check by uncommenting the following lines
# #print("After reordering for clustering: ") 
# #print(samples_df)
# #print("This is the new index: ")
# #print(ind_sorted)

In [49]:
# # list of labeled indexes 
# # convert Index class to np.array and then to list of strings
# sampind_list = ind_sorted.values.tolist()
# #print(sampind_list)

In [50]:
# # load all .filteredQualityReadCounts files
# # collect 'filtered' readcounts into array of dataframes 
# readcounts_df = []
# for sample in ind_sorted: 
#     # here we alter the code to read ".filteredQualityReadCounts" files 
#     df = pd.read_csv("./SpermAnalysis_Muller1/data/viz-Output/" + sample + "/" + sample + ".filteredQualityReadCounts", delimiter="\t", usecols=[0,2,3])
#     #print(df)
#     readcounts_df.append(df)

In [51]:
# # extract event and count columns from each sample to create one big dataframe of all samples 
# array_sampdf = []
# for ind,df in enumerate(readcounts_df): 
#     sample_id = ind_sorted[ind]
#     sampdf = df[['event','count']]
#     sampdf = sampdf.set_index('event')
#     shortID = re.sub(r'-sperm', '_S', sample_id)
#     sampdf = sampdf.rename(columns={'count':f'{shortID}'})
#     array_sampdf.append(sampdf)

### Write composite barcode matrix file 
###### You'll use this for generation of Muller plots

In [52]:
# # join dataframes together along the column axis, taking the union of indices 
# newdf = pd.concat(array_sampdf, axis=1, sort=False)
# newdf.index.name = 'barcode'
# newdf.to_csv(experiments_subfolder+"barcodeMatrix_3filters.tsv", sep="\t") # save for ease of viewing

### Write barcode matrix file for each fish in data set

In [53]:
# # collect list of fish, each will get its own barcode matrix file
# # saved each fish name as a key in a dictionary with an empty list for the value
# fish_names = {}
# for fish in sampind_list: 
#     fish = fish.split("_")
#     fish_name = fish[0]
#     if fish_name not in fish_names: 
#         fish_names[fish_name] = []

In [54]:
# # for all the keys in fish_names, populate its empty list with the matrices associated with that fish name
# for fish_name in fish_names.keys():
#     for sample in array_sampdf:
#         sample_name = str(sample).split("_")[0].split(" ")[-1]
#         if fish_name == sample_name: 
#             fish_names[fish_name].append(sample)

### Step E summary report

In [55]:
summary = open(experiments_subfolder+"stepE_summary.txt", 'w')
summary.write("Summary of step E: A barcode matrix file is written to the experiments folder and\nis used for generating Muller plots.\n\n")
summary.write("A barcode matrix file is also written for each fish in your samples.\n")
summary.write("The barcodes present in these matrix files will have passed all three filters (steps A, B and D)")
summary.write("\n")
summary.close()

# Job report

In [56]:
summary = open(experiments_subfolder+"full_job_report.txt", 'w')
stepA = open(experiments_subfolder+"stepA_summary.txt", 'r')
stepB = open(experiments_subfolder+"stepB_summary.txt", 'r')
# stepC = open(experiments_subfolder+"stepC_summary.txt", 'r')
stepD = open(experiments_subfolder+"stepD_summary.txt", 'r')
stepE = open(experiments_subfolder+"stepE_summary.txt", 'r')

summary.write("Processed GESTALT barcodes from serially sampled sperm data.\n")
summary.write("Your job completed at " + datetime.datetime.now().strftime('%I:%M %p on %m/%d/%Y'))
summary.write("\nAndy Sposato wrote original code for steps A-D.\nAndy adapted the code in Step E from original code written by Kimberly Truong.")
summary.write("\n\n############################################################################################\n\n")
for line in stepA.readlines(): 
    summary.write(line)
stepA.close()
summary.write("\n############################################################################################\n\n")
for line in stepB.readlines(): 
    summary.write(line)
stepB.close()
# summary.write("############################################################################################\n\n")
# for line in stepC.readlines(): 
#     summary.write(line)
# stepC.close()
summary.write("############################################################################################\n\n")
for line in stepD.readlines(): 
    summary.write(line)
stepD.close()
summary.write("############################################################################################\n\n")
for line in stepE.readlines(): 
    summary.write(line)
stepE.close()
summary.write("\n############################################################################################\n\n")
summary.write("It was a pleasure filtering your sperm barcodes. \n")
summary.close()